# Eye Closure Detection with EAR

Explore one image, calibrate a personal threshold, then compare short and sustained closures in video.

In [ ]:
import cv2
import matplotlib.pyplot as plt

camera = cv2.VideoCapture(0)

try:
    if not camera.isOpened():
        raise RuntimeError("Could not open the camera.")

    for i in range(30):
        success, image = camera.read()
        if not success:
            raise RuntimeError("Could not capture an image.")
finally:
    camera.release()

if success:

    image = cv2.flip(image, 1)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.imshow(image_rgb)
    plt.axis("off")
    plt.show()
else:
    print("Could not capture an image.")


In [ ]:
type(image)


In [ ]:
image.shape
height = image.shape[0]
width = image.shape[1]


In [ ]:
half_height = height // 2
half_width = width // 2

image_rgb[0:half_height, 0:half_width]


In [ ]:
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


In [ ]:
model_options = python.BaseOptions(model_asset_path="models/face_landmarker.task")
model_options.model_asset_path


In [ ]:
configuration = vision.FaceLandmarkerOptions(
    base_options=model_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1
)

detector = vision.FaceLandmarker.create_from_options(configuration)


In [ ]:
mp_image = mp.Image(
    image_format = mp.ImageFormat.SRGB,
    data = image_rgb
)


In [ ]:
result = detector.detect(mp_image)
print("Faces detected:", len(result.face_landmarks))


In [ ]:
if not result.face_landmarks:
    raise ValueError("No face detected.")

landmarks = result.face_landmarks[0]
height, width = image_rgb.shape[:2]

output_image = image_rgb.copy()

for landmark in landmarks:
    x_pixel = landmark.x * width
    y_pixel = landmark.y * height

    center = (int(x_pixel), int(y_pixel))
    cv2.circle(output_image, center, 2, (255, 0, 0), -1)

plt.imshow(output_image)
plt.axis("off")
plt.show()


In [ ]:
import math

eye_a_indices = [33, 160, 158, 133, 153, 144]
eye_b_indices = [362, 385, 387, 263, 373, 380]

def dist(a, b):
    difference_x = a[0] - b[0]
    difference_y = a[1] - b[1]

    return math.sqrt(difference_x**2 + difference_y**2)

def calculate_ear(eye):
    vertical_1 = dist(eye[1], eye[5])
    vertical_2 = dist(eye[2], eye[4])
    horizontal = dist(eye[0], eye[3])

    if horizontal < 1e-6:
        raise ValueError("Cannot calculate EAR: eye width is nearly zero.")

    return (vertical_1 + vertical_2) / (2 * horizontal)

def get_coordinates(indices, landmarks, width, height):
    coordinates = []

    for index in indices:
        landmark = landmarks[index]

        x_pixel = landmark.x * width
        y_pixel = landmark.y * height

        coordinates.append((x_pixel, y_pixel))

    return coordinates

eye_a_coordinates = get_coordinates(eye_a_indices, landmarks, width, height )

eye_b_coordinates = get_coordinates(eye_b_indices, landmarks, width, height)

ear_a = calculate_ear(eye_a_coordinates)
ear_b = calculate_ear(eye_b_coordinates)
mean_ear = (ear_a + ear_b) / 2

print("Eye A EAR:", ear_a)
print("Eye B EAR:", ear_b)
print("Mean EAR:", mean_ear)

eye_image = image_rgb.copy()

for coordinates, color in [
    (eye_a_coordinates, (255, 0, 0)),
    (eye_b_coordinates, (0, 255, 0))
]:
    for x_pixel, y_pixel in coordinates:
        center = (int(x_pixel), int(y_pixel))
        cv2.circle(eye_image, center, 3, color, -1)

plt.imshow(eye_image)
plt.axis("off")
plt.show()


## Calibrate the threshold

Follow the two prompts. Keep the same position and lighting for both batches.

In [ ]:
import time

EAR_open = []
EAR_closed = []

for condition, samples in [("open", EAR_open), ("closed", EAR_closed)]:
    input(f"Press Enter to prepare for the {condition}-eye batch. ")
    camera = cv2.VideoCapture(0)

    try:
        if not camera.isOpened():
            raise RuntimeError("Could not open the camera.")

        print(f"Get ready: keep your eyes {condition}.")
        print("The batch will begin in 3 seconds.", flush=True)
        time.sleep(3)

        for number in range(1, 6):

            start = time.monotonic()

            while True:
                success, capture = camera.read()

                if not success:
                    raise RuntimeError("Could not read the camera.")

                if time.monotonic() - start >= 1:
                    break

            capture = cv2.flip(capture, 1)
            capture_rgb = cv2.cvtColor(capture, cv2.COLOR_BGR2RGB)
            current_height, current_width = capture_rgb.shape[:2]

            capture_mp = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=capture_rgb
            )

            current_result = detector.detect(capture_mp)

            if not current_result.face_landmarks:
                print(f"Photo {number}/5: no face; not saved.")
                continue

            current_landmarks = current_result.face_landmarks[0]

            eye_a = get_coordinates(
                eye_a_indices, current_landmarks, current_width, current_height
            )
            eye_b = get_coordinates(
                eye_b_indices, current_landmarks, current_width, current_height
            )

            try:
                ear = (calculate_ear(eye_a) + calculate_ear(eye_b)) / 2
            except ValueError:
                print(f"Photo {number}/5: invalid geometry; not saved.")
                continue

            samples.append(ear)
            print(f"Photo {number}/5: EAR = {ear:.4f}", flush=True)

    finally:
        camera.release()

print("Total open:", len(EAR_open))
print("Total closed:", len(EAR_closed))


In [ ]:
threshold = None

if not EAR_open or not EAR_closed:
    raise ValueError("Collect valid samples from both calibration batches first.")

plt.scatter(
    range(len(EAR_open)), EAR_open, label="Open"
)
plt.scatter(
    range(len(EAR_closed)), EAR_closed, label="Closed"
)

plt.xlabel("Sample number")
plt.ylabel("Mean EAR")
plt.legend()
plt.show()

print("Open range:", min(EAR_open), max(EAR_open))
print("Closed range:", min(EAR_closed), max(EAR_closed))

p10_open = np.percentile(EAR_open, 10)
p90_closed = np.percentile(EAR_closed, 90)

if p90_closed < p10_open:
    threshold = float((p10_open + p90_closed) / 2)
    print("Provisional threshold:", threshold)
else:
    raise ValueError("Open and closed samples overlap. Repeat calibration.")


Because EAR decreases when the eyes close, values below the threshold calibrated on personal samples are classified as closed; values at or above it are classified as open.


## Record and analyze a sequence

Keep your eyes visible while the notebook records EAR for 10 seconds.

In [ ]:
if threshold is None:
    raise RuntimeError("Complete calibration before starting video capture.")

import time
import numpy as np

video_options = vision.FaceLandmarkerOptions(
    base_options=model_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1
)

video_detector = vision.FaceLandmarker.create_from_options(
    video_options
)

times = []
ear_values = []

duration = 10

camera = cv2.VideoCapture(0)

if not camera.isOpened():
    camera.release()
    video_detector.close()
    raise RuntimeError("Could not open the camera.")

start_time = time.monotonic()

previous_timestamp = -1

try:
    while True:

        current_time = time.monotonic() - start_time

        if current_time >= duration:
            break

        success, frame = camera.read()

        if not success:
            print("Could not read a frame.")
            break

        frame = cv2.flip(frame, 1)

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        current_height, current_width = frame_rgb.shape[:2]

        frame_mp = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame_rgb
        )

        timestamp_ms = int(current_time * 1000)

        timestamp_ms = max(
            previous_timestamp + 1,
            timestamp_ms
        )

        previous_timestamp = timestamp_ms

        video_result = video_detector.detect_for_video(
            frame_mp,
            timestamp_ms
        )

        if video_result.face_landmarks:

            current_landmarks = video_result.face_landmarks[0]

            left_eye_coordinates = get_coordinates(
                eye_a_indices,
                current_landmarks,
                current_width,
                current_height
            )

            right_eye_coordinates = get_coordinates(
                eye_b_indices,
                current_landmarks,
                current_width,
                current_height
            )

            try:
                left_ear = calculate_ear(left_eye_coordinates)
                right_ear = calculate_ear(right_eye_coordinates)
                current_ear = (left_ear + right_ear) / 2
            except ValueError:
                current_ear = np.nan

        else:

            current_ear = np.nan

        times.append(current_time)
        ear_values.append(current_ear)

finally:

    camera.release()

    video_detector.close()

print("Sequence finished.")
print("Number of samples:", len(ear_values))


In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(times, ear_values, label="EAR")

plt.axhline(
    threshold,
    color="red",
    linestyle="--",
    label=f"Threshold = {threshold:.3f}"
)

plt.xlabel("Time (seconds)")
plt.ylabel("EAR")
plt.title("EAR during a video sequence")
plt.legend()
plt.grid()
plt.show()


In [ ]:
isitopenorclosed = []

aux_time_just_opened = [0] * len(times)
aux_time_just_closed = [0] * len(times)

closed_start_time = 0

if not ear_values:
    raise ValueError("The sequence contains no samples.")

if np.isnan(ear_values[0]):
    isitopenorclosed.append("NO FACE")
elif ear_values[0] < threshold:
    isitopenorclosed.append("CLOSED")
    closed_start_time = times[0]
else:
    isitopenorclosed.append("OPEN")

for i in range(1 , len(times)):

    if np.isnan(ear_values[i]):
        isitopenorclosed.append("NO FACE")
        closed_start_time = 0

    elif ear_values[i] < threshold:
        isitopenorclosed.append("CLOSED")

        if isitopenorclosed[i-1] != "CLOSED":
            closed_start_time = times[i]
            aux_time_just_closed[i] = times[i]

    else:
        isitopenorclosed.append("OPEN")

        if isitopenorclosed[i-1] == "CLOSED":
            closed_eyes_duration = times[i] - closed_start_time
            aux_time_just_opened[i] = closed_eyes_duration
            closed_start_time = 0

if isitopenorclosed[-1] == "CLOSED":
    closed_eyes_duration = times[-1] - closed_start_time
    aux_time_just_opened[-1] = closed_eyes_duration

print(isitopenorclosed)

print("Closed durations:", [value for value in aux_time_just_opened if value > 0])


In [ ]:
minimum_duration = 1.0

for duration in aux_time_just_opened:

    if duration == 0:
        continue

    print("The eyes were closed for", round(duration, 2), "seconds")

    if duration < minimum_duration:
        print("This was a short closure")
    else:
        print("This was a long closure")


In [ ]:
states = []

for i in range(len(isitopenorclosed)):
    if isitopenorclosed[i] == "OPEN":
        states.append(1)
    elif isitopenorclosed[i] == "CLOSED":
        states.append(0)
    else:
        states.append(np.nan)

print(states)
print(len(states))
print(len(times))


In [ ]:
plt.figure(figsize=(12, 4))

plt.step(
    times,
    states,
    where="post"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Eye state")

plt.yticks(
    [0, 1],
    ["Closed", "Open"]
)

plt.title("Eye state during the sequence")
plt.grid()
plt.show()


In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(times, ear_values, label="EAR")

plt.axhline(
    threshold,
    color="red",
    linestyle="--",
    label=f"Threshold = {threshold:.3f}"
)

plt.xlabel("Time (seconds)")
plt.ylabel("EAR")
plt.title("EAR during the sequence")
plt.legend()
plt.grid()
plt.show()

print("Minimum EAR:", np.nanmin(ear_values))
print("Maximum EAR:", np.nanmax(ear_values))
print("Threshold:", threshold)


In [ ]:
print("Closed durations:")

for duration in aux_time_just_opened:

    if duration > 0:
        print(round(duration, 2), "seconds")

minimum_duration = 1.0

print("Classification:")

for duration in aux_time_just_opened:

    if duration > 0:

        if duration < minimum_duration:
            print(
                round(duration, 2),
                "seconds -> short closure"
            )
        else:
            print(
                round(duration, 2),
                "seconds -> prolonged closure"
            )


In [ ]:
minimum_duration = 1.0

alert = False
alert_durations = []

for duration in aux_time_just_opened:

    if duration >= minimum_duration:
        alert = True
        alert_durations.append(duration)

if alert:
    print("ALERT: prolonged eye closure detected")

    for duration in alert_durations:
        print(
            "Closure duration:",
            round(duration, 2),
            "seconds"
        )
else:
    print("No prolonged eye closure detected")


## Live alert

Run the cell to open the webcam display. With the camera window focused, press `q` to stop.

In [ ]:
if threshold is None:
    raise RuntimeError("Complete calibration before starting video capture.")

import time
import numpy as np
import subprocess

minimum_duration = 1.0

video_options = vision.FaceLandmarkerOptions(
    base_options=model_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1
)

video_detector = vision.FaceLandmarker.create_from_options(
    video_options
)

camera = cv2.VideoCapture(0)

if not camera.isOpened():
    camera.release()
    video_detector.close()
    raise RuntimeError("Could not open the camera.")

start_time = time.monotonic()
previous_timestamp = -1
closed_start_time = None
previous_alert = False
sound_process = None

try:
    while True:

        success, frame = camera.read()

        if not success:
            print("Could not read a frame.")
            break

        current_time = time.monotonic() - start_time

        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        height, width = frame_rgb.shape[:2]

        frame_mp = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame_rgb
        )

        timestamp_ms = max(
            previous_timestamp + 1,
            int(current_time * 1000)
        )

        previous_timestamp = timestamp_ms

        result = video_detector.detect_for_video(
            frame_mp,
            timestamp_ms
        )

        current_ear = np.nan

        if result.face_landmarks:

            landmarks = result.face_landmarks[0]

            eye_a = get_coordinates(
                eye_a_indices,
                landmarks,
                width,
                height
            )

            eye_b = get_coordinates(
                eye_b_indices,
                landmarks,
                width,
                height
            )

            try:
                ear_a = calculate_ear(eye_a)
                ear_b = calculate_ear(eye_b)
                current_ear = (ear_a + ear_b) / 2
            except ValueError:
                current_ear = np.nan

        if np.isnan(current_ear):

            state = "NO FACE"
            closed_start_time = None
            alert = False

        elif current_ear < threshold:

            state = "CLOSED"

            if closed_start_time is None:
                closed_start_time = current_time

            closed_duration = current_time - closed_start_time
            alert = closed_duration >= minimum_duration

        else:

            state = "OPEN"
            closed_start_time = None
            closed_duration = 0
            alert = False

        if alert and not previous_alert:
            if sound_process is None or sound_process.poll() is not None:
                sound_process = subprocess.Popen([
                    "afplay", "/System/Library/Sounds/Glass.aiff"
                ])
        previous_alert = alert

        if np.isnan(current_ear):
            ear_text = "EAR: --"
        else:
            ear_text = f"EAR: {current_ear:.3f}"

        cv2.putText(
            frame,
            ear_text,
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"State: {state}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        if alert:

            cv2.putText(
                frame,
                "ALERT: PROLONGED CLOSURE",
                (20, 115),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                3
            )

        cv2.imshow("Eye closure detector", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    camera.release()
    video_detector.close()
    cv2.destroyAllWindows()
    if sound_process is not None and sound_process.poll() is None:
        sound_process.terminate()
